# Static Analysis in Practice — Custom Semgrep Rules for ML Security

This notebook demonstrates how to **write and test custom Semgrep rules** for AI/ML security patterns:

- Unsafe `pickle.load()`
- Unsafe `torch.load()`
- Hardcoded API keys
- Path traversal in model loading
- Missing data validation

It mirrors the *“Static Analysis in Practice – Custom Semgrep Rules Demo”* video.

## 1. Create Sample Vulnerable Files

We create small Python files that intentionally contain ML‑specific vulnerabilities.
These will be scanned by our custom Semgrep rules.

In [ ]:
import os
import subprocess
import json

print("="*70)
print("CUSTOM SEMGREP RULES FOR ML SECURITY")
print("="*70)

# Sample 1: Unsafe pickle usage
sample1_pickle = '''
import pickle

def load_model(model_path):
    """Vulnerable: pickle.load() with user input"""
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    return model

def load_trusted_model():
    """Safe: hardcoded trusted path"""
    with open('/trusted/models/production.pkl', 'rb') as f:
        model = pickle.load(f)
    return model
'''

# Sample 2: Unsafe PyTorch loading
sample2_pytorch = '''
import torch

def load_pytorch_model(path):
    """Vulnerable: torch.load() without weights_only"""
    model = torch.load(path)
    return model

def load_pytorch_safe(path):
    """Safe: uses weights_only parameter"""
    model = torch.load(path, weights_only=True)
    return model
'''

# Sample 3: Hardcoded credentials
sample3_credentials = '''
import os

# Vulnerable: hardcoded API keys
API_KEY = "sk-1234567890abcdefghijklmnop"
OPENAI_KEY = "api_key_abcdefghijklmnopqrstuvwxyz"

def get_config():
    return {
        'api_key': 'sk-proj-test1234567890',
        'secret': 'my-secret-token'
    }

# Safe: from environment
SAFE_KEY = os.getenv('API_KEY')
'''

# Sample 4: Path traversal
sample4_path_traversal = '''
from flask import request
import pickle

def predict(user_model_name):
    """Vulnerable: user input in file path"""
    model_path = f'models/{user_model_name}.pkl'
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    return model.predict([1, 2, 3])

def predict_safe(model_id):
    """Safe: validated against whitelist"""
    allowed_models = ['model1', 'model2', 'model3']
    if model_id not in allowed_models:
        raise ValueError("Invalid model")
    return load_model(f'models/{model_id}.pkl')
'''

# Sample 5: Missing data validation
sample5_data_validation = '''
import os
import pandas as pd

def load_training_data(file_path):
    """Vulnerable: no validation"""
    data = pd.read_csv(file_path)
    return data

def load_training_data_safe(file_path):
    """Safe: with validation"""
    if not os.path.exists(file_path):
        raise FileNotFoundError()
    data = pd.read_csv(file_path)
    if data.shape[0] > 1000000:
        raise ValueError("File too large")
    return data
'''

samples = {
    'sample1_pickle.py': sample1_pickle,
    'sample2_pytorch.py': sample2_pytorch,
    'sample3_credentials.py': sample3_credentials,
    'sample4_path_traversal.py': sample4_path_traversal,
    'sample5_data_validation.py': sample5_data_validation
}

for filename, content in samples.items():
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)

print("\n✓ Created sample vulnerable files for testing")

## 2. Define Custom Semgrep Rules (ml-security-rules.yml)

We now create a **Semgrep rule file** that encodes ML‑specific security patterns.

These rules are exactly what you’d use in CI/CD or local scans.

In [ ]:
semgrep_rules = '''
rules:
  - id: unsafe-pickle-load
    pattern: pickle.load($ARG)
    message: "Unsafe pickle deserialization detected. Pickle can execute arbitrary code. Use safer alternatives like joblib, ONNX, or SafeTensors."
    languages: [python]
    severity: ERROR
    metadata:
      category: security
      cwe: "CWE-502: Deserialization of Untrusted Data"
      confidence: HIGH
      references:
        - "https://owasp.org/www-community/vulnerabilities/Deserialization_of_untrusted_data"

  - id: unsafe-pickle-with-user-input
    patterns:
      - pattern-inside: |
          def $FUNC(..., $USER_INPUT, ...):
            ...
      - pattern: pickle.load(...)
    message: "Critical: pickle.load() with user-controlled input. This allows remote code execution."
    languages: [python]
    severity: ERROR
    metadata:
      category: security
      cwe: "CWE-502"
      confidence: HIGH

  - id: unsafe-torch-load
    patterns:
      - pattern: torch.load($PATH, ...)
      - pattern-not: torch.load($PATH, ..., weights_only=True, ...)
    message: "Unsafe torch.load() without weights_only=True. This can execute arbitrary code. Use torch.load(path, weights_only=True)"
    languages: [python]
    severity: WARNING
    metadata:
      category: security
      cwe: "CWE-502"
      references:
        - "https://pytorch.org/docs/stable/generated/torch.load.html"

  - id: hardcoded-api-key
    patterns:
      - pattern-either:
          - pattern: $VAR = "sk-..."
          - pattern: $VAR = "api_key_..."
          - pattern: $VAR = "sk_..."
      - metavariable-regex:
          metavariable: $VAR
          regex: (?i)(key|token|secret|password|api)
    message: "Hardcoded API key detected. Store credentials in environment variables or use a secrets manager."
    languages: [python]
    severity: ERROR
    metadata:
      category: security
      cwe: "CWE-798: Use of Hard-coded Credentials"
      confidence: HIGH

  - id: path-traversal-in-model-loading
    patterns:
      - pattern-inside: |
          def $FUNC(..., $USER_INPUT, ...):
            ...
      - pattern-either:
          - pattern: open($USER_INPUT, ...)
          - pattern: open(f"...{$USER_INPUT}...", ...)
      - pattern-not-inside: |
          if $USER_INPUT in $WHITELIST:
            ...
    message: "Potential path traversal vulnerability. User input used in file path without validation."
    languages: [python]
    severity: ERROR
    metadata:
      category: security
      cwe: "CWE-22: Path Traversal"

  - id: missing-data-validation
    patterns:
      - pattern: $DATA = pd.read_csv($PATH)
      - pattern-not-inside: |
          if os.path.exists($PATH):
            ...
      - pattern-not-inside: |
          if $DATA.shape[0] > $N:
            ...
    message: "CSV data loaded without validation. Add file existence checks, size limits, and schema validation."
    languages: [python]
    severity: WARNING
    metadata:
      category: security
      best-practice: "Always validate data sources"
'''

with open('ml-security-rules.yml', 'w', encoding='utf-8') as f:
    f.write(semgrep_rules)

print("✓ Created custom Semgrep rules: ml-security-rules.yml")

## 3. Helper Function to Run Semgrep

We define a small wrapper to run Semgrep and parse JSON output.

If Semgrep is not installed, we print expected findings instead.

In [ ]:
def run_semgrep(rule_file, target_files):
    """Run Semgrep with custom rules and return parsed JSON findings."""
    try:
        cmd = ['semgrep', '--config', rule_file, '--json'] + target_files
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30,
            encoding='utf-8',
            errors='replace'
        )
        if result.returncode in (0, 1):
            try:
                findings = json.loads(result.stdout)
                return findings
            except Exception:
                return None
        return None
    except FileNotFoundError:
        print("❌ Semgrep not installed. Install with: pip install semgrep")
        return None
    except Exception as e:
        print(f"Error running Semgrep: {e}")
        return None

## 4. Run Semgrep on Sample Files

We now scan all sample files with our custom rules and display findings grouped by file.

In [ ]:
print("\n" + "="*70)
print("RUNNING SEMGREP WITH CUSTOM RULES")
print("="*70)

print("\n[Scanning Sample Files]\n")
all_files = list(samples.keys())
findings = run_semgrep('ml-security-rules.yml', all_files)

if findings and 'results' in findings:
    results_by_file = {}
    for result in findings['results']:
        filepath = result['path']
        results_by_file.setdefault(filepath, []).append(result)
    
    for filepath, file_results in results_by_file.items():
        print(f"\n📄 {filepath}")
        print("-" * 70)
        for r in file_results:
            rule_id = r['check_id'].split('.')[-1]
            line = r['start']['line']
            severity = r['extra']['severity']
            message = r['extra']['message']
            emoji = "🔴" if severity == "ERROR" else "🟡"
            print(f"{emoji} Line {line}: {rule_id}")
            print(f"   {message}")
        print()
    
    total_errors = sum(1 for r in findings['results'] if r['extra']['severity'] == 'ERROR')
    total_warnings = sum(1 for r in findings['results'] if r['extra']['severity'] == 'WARNING')
    
    print("="*70)
    print("SCAN SUMMARY")
    print("="*70)
    print(f"Total findings: {len(findings['results'])}")
    print(f"  Errors (🔴): {total_errors}")
    print(f"  Warnings (🟡): {total_warnings}")
    print(f"Files scanned: {len(all_files)}")
else:
    print("\nNote: Install Semgrep to see scan results: pip install semgrep")
    print("\nExpected findings:")
    print("  • sample1_pickle.py: unsafe pickle.load() (2 cases)")
    print("  • sample2_pytorch.py: unsafe torch.load()")
    print("  • sample3_credentials.py: hardcoded API keys")
    print("  • sample4_path_traversal.py: path traversal in model loading")
    print("  • sample5_data_validation.py: missing data validation")

## 5. Summary of Custom Rules

We’ve created a reusable **ML security rulepack**:

- `unsafe-pickle-load`
- `unsafe-pickle-with-user-input`
- `unsafe-torch-load`
- `hardcoded-api-key`
- `path-traversal-in-model-loading`
- `missing-data-validation`

You can now:
- Run locally: `semgrep --config ml-security-rules.yml .`
- Add to CI/CD: GitHub Actions, GitLab CI, etc.
- Use in IDE: VS Code / PyCharm Semgrep integrations.


In [ ]:
print("\n" + "="*70)
print("COMPLETE RULESET CREATED")
print("="*70)
print("\nCustom rules in ml-security-rules.yml:")
print("  1. unsafe-pickle-load")
print("  2. unsafe-pickle-with-user-input")
print("  3. unsafe-torch-load")
print("  4. hardcoded-api-key")
print("  5. path-traversal-in-model-loading")
print("  6. missing-data-validation")
print("\nHow to use:")
print("  • Local: semgrep --config ml-security-rules.yml .")
print("  • CI/CD: add to your pipeline config")
print("  • IDE: configure Semgrep plugin to use this rule file")

# Optional cleanup of sample files (keep if you want to re-run)
for filename in samples.keys():
    if os.path.exists(filename):
        os.remove(filename)
print("\n(Optional) Sample files cleaned up.")